# Deep-learning avalanche detection vs. classical image processing

Trains a U-Net (ResNet-34 encoder, ImageNet-pretrained) on the expert-annotated SLF
avalanche webcam dataset ([Hafner 2024, EnviDat, CC BY-SA 4.0](https://doi.org/10.16904/envidat.526))
and compares it with the classical pipeline of this repository on the same image pairs.

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
It takes about 30–40 minutes (most of it the 2.9 GB download and the training).

**Honest evaluation:** cameras are split into 3 folds (*leave-camera-view-out*).
Every image is evaluated by a model that never saw that camera view during training.

In [ ]:
# 1) Setup
import subprocess, torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))
!pip -q install segmentation-models-pytorch
![ -d /content/snow-detection ] || git clone -q https://github.com/bircanbektas/snow-detection.git /content/snow-detection
%cd /content/snow-detection

In [ ]:
# 2) Download the SLF dataset (~2.9 GB) and extract only the images and masks
import re, zipfile
from pathlib import Path

URL = ("https://www.envidat.ch/dataset/85ae804d-3282-4935-82b2-9e43d024c266/resource/"
       "e0d0a419-86e9-4fde-9254-410851363b80/download/ias_data.zip")
ZIP, RAW = Path("/content/ias_data.zip"), Path("/content/slf_raw")
if not ZIP.exists():
    print("Downloading, this takes a few minutes...")
    subprocess.run(["wget", "-q", "-O", str(ZIP), URL], check=True)
SPLITS = ("Train", "Vali", "Test", "WebNew", "GroundPic", "UserPic")
pattern = re.compile(r"(^|/)(%s)/(images|masks)/[^/]+$" % "|".join(SPLITS))
if not RAW.exists():
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(RAW, [m for m in z.namelist() if pattern.search(m)])
ROOT = next(RAW.rglob("Train")).parent
for s in SPLITS:
    print(s, len(list((ROOT / s / "images").glob("*"))), "files")

In [ ]:
# 3) Prepare: one image per photo with all its avalanches merged into one mask, width 1200 px.
#    Webcam images (Train/Vali/Test/WebNew) are grouped by camera view with the repo script;
#    the hand-held photos (GroundPic/UserPic) are only used for training.
import cv2, numpy as np
from collections import defaultdict

!python evaluation/prepare_slf.py {ROOT} /content/prepared

EXTRA = Path("/content/extra")
EXTRA.mkdir(exist_ok=True)
WIDTH = 1200
for split in ("GroundPic", "UserPic"):
    groups = defaultdict(lambda: {"image": None, "masks": []})
    masks = {m.stem: m for m in (ROOT / split / "masks").glob("*")}
    for img in (ROOT / split / "images").glob("*"):
        key = split + "_" + re.sub(r"_\d+$", "", img.stem)
        groups[key]["image"] = img
        if img.stem in masks:
            groups[key]["masks"].append(masks[img.stem])
    used = 0
    for key, g in groups.items():
        img = cv2.imread(str(g["image"]))
        if img is None or not g["masks"]:
            continue
        scale = WIDTH / img.shape[1]
        small = cv2.resize(img, (WIDTH, int(img.shape[0] * scale)), interpolation=cv2.INTER_AREA)
        union = np.zeros(small.shape[:2], np.uint8)
        for m in g["masks"]:
            mk = cv2.imread(str(m), cv2.IMREAD_GRAYSCALE)
            if mk is not None:
                union[cv2.resize(mk, small.shape[1::-1], interpolation=cv2.INTER_NEAREST) > 0] = 255
        cv2.imwrite(str(EXTRA / f"{key}.jpg"), small, [cv2.IMWRITE_JPEG_QUALITY, 92])
        cv2.imwrite(str(EXTRA / f"{key}_mask.png"), union)
        used += 1
    print(f"{split}: {used} photos with masks")

In [ ]:
# 4) Data, model and training code
import json, random, time
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

PREP = Path("/content/prepared")
idx = json.load(open(PREP / "index.json"))
webcam = [it["key"] for it in idx["items"]]
extra = sorted(p.stem for p in EXTRA.glob("*.jpg"))


def load(folder, key):
    img = cv2.imread(str(folder / f"{key}.jpg"))
    mask = (cv2.imread(str(folder / f"{key}_mask.png"), 0) > 0).astype(np.float32)
    return img, mask


data = {k: load(PREP, k) for k in webcam}
data.update({k: load(EXTRA, k) for k in extra})
print(len(webcam), "webcam images,", len(extra), "hand-held photos")

MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)
CROP = 512


def to_tensor(img):
    rgb = (img[..., ::-1].astype(np.float32) / 255 - MEAN) / STD
    return torch.from_numpy(rgb.transpose(2, 0, 1).copy())


def random_crop(img, mask):
    s = random.uniform(0.6, 1.2)
    img = cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    mask = cv2.resize(mask, img.shape[1::-1], interpolation=cv2.INTER_NEAREST)
    ph, pw = max(CROP - img.shape[0], 0), max(CROP - img.shape[1], 0)
    if ph or pw:
        img = cv2.copyMakeBorder(img, 0, ph, 0, pw, cv2.BORDER_REFLECT)
        mask = cv2.copyMakeBorder(mask, 0, ph, 0, pw, cv2.BORDER_CONSTANT, value=0)
    h, w = mask.shape
    ys, xs = np.nonzero(mask)
    if len(ys) and random.random() < 0.6:          # avalanches are small: often crop around one
        i = random.randrange(len(ys))
        y0 = ys[i] - CROP // 2 + random.randint(-CROP // 4, CROP // 4)
        x0 = xs[i] - CROP // 2 + random.randint(-CROP // 4, CROP // 4)
    else:
        y0, x0 = random.randint(0, h - CROP), random.randint(0, w - CROP)
    y0, x0 = int(np.clip(y0, 0, h - CROP)), int(np.clip(x0, 0, w - CROP))
    img, mask = img[y0:y0 + CROP, x0:x0 + CROP], mask[y0:y0 + CROP, x0:x0 + CROP]
    if random.random() < 0.5:
        img, mask = img[:, ::-1], mask[:, ::-1]
    # lighting changes: contrast, brightness, gamma
    img = img.astype(np.float32) * random.uniform(0.7, 1.3) + random.uniform(-30, 30)
    img = 255 * (np.clip(img, 0, 255) / 255) ** random.uniform(0.7, 1.4)
    return img.astype(np.uint8), mask


class Crops(Dataset):
    def __init__(self, keys, per_image=4):
        self.keys, self.per_image = keys, per_image

    def __len__(self):
        return len(self.keys) * self.per_image

    def __getitem__(self, i):
        img, mask = random_crop(*data[self.keys[i % len(self.keys)]])
        return to_tensor(img), torch.from_numpy(mask.copy())[None]


def loss_fn(logits, y):
    p = torch.sigmoid(logits)
    dice = 1 - (2 * (p * y).sum() + 1) / (p.sum() + y.sum() + 1)
    return F.binary_cross_entropy_with_logits(logits, y) + dice


def train(keys, epochs=30, seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = smp.Unet("resnet34", encoder_weights="imagenet", classes=1).cuda()
    dl = DataLoader(Crops(keys), batch_size=8, shuffle=True, num_workers=2, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=epochs * len(dl))
    scaler = torch.amp.GradScaler()
    start = time.time()
    for ep in range(epochs):
        model.train()
        total = 0
        for x, y in dl:
            x, y = x.cuda(), y.cuda()
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(x)
            loss = loss_fn(logits.float(), y)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            total += loss.item()
        if (ep + 1) % 5 == 0:
            print(f"  epoch {ep + 1}/{epochs}  loss {total / len(dl):.3f}  {time.time() - start:.0f}s")
    return model.eval()


@torch.no_grad()
def predict(model, img):
    """Avalanche probability per pixel (average of the image and its mirror)."""
    h, w = img.shape[:2]
    padded = cv2.copyMakeBorder(img, 0, (-h) % 32, 0, (-w) % 32, cv2.BORDER_REFLECT)
    x = to_tensor(padded)[None].cuda()
    with torch.autocast("cuda", dtype=torch.float16):
        p = torch.sigmoid(model(x)) + torch.sigmoid(model(x.flip(-1))).flip(-1)
    return (p[0, 0, :h, :w] / 2).float().cpu().numpy()

In [ ]:
# 5) Leave-camera-view-out training: 3 folds, each model is tested on views it never saw
view_of = {k: vi for vi, v in enumerate(idx["views"]) for k in v}
groups = defaultdict(list)
for k in webcam:
    groups[view_of.get(k, "single_" + k)].append(k)
folds = [[], [], []]
for g in sorted(groups.values(), key=len, reverse=True):
    min(folds, key=len).extend(g)

prob = {}
for fi, test_keys in enumerate(folds):
    train_keys = [k for k in webcam if k not in test_keys] + extra
    print(f"Fold {fi + 1}/3: training on {len(train_keys)} images, testing on {len(test_keys)}")
    model = train(train_keys)
    for k in test_keys:
        prob[k] = predict(model, data[k][0])
    del model
    torch.cuda.empty_cache()

In [ ]:
# 6) Per-image segmentation quality on unseen camera views
tp = fp = fn = 0
ious = []
for k in webcam:
    pred, gt = prob[k] > 0.5, data[k][1] > 0
    a, b, c = np.sum(pred & gt), np.sum(pred & ~gt), np.sum(~pred & gt)
    tp, fp, fn = tp + a, fp + b, fn + c
    if gt.any():
        ious.append(a / (a + b + c))
seg = {"pixel_iou": tp / (tp + fp + fn), "pixel_f1": 2 * tp / (2 * tp + fp + fn),
       "pixel_precision": tp / max(tp + fp, 1), "pixel_recall": tp / max(tp + fn, 1),
       "mean_image_iou": float(np.mean(ious))}
for name, v in seg.items():
    print(f"{name:>16}: {v:.3f}")

In [ ]:
# 7) Same pair evaluation as evaluation/evaluate_slf.py: classical pipeline vs. model
import contextlib, io
from datetime import datetime
from avalanche_monitor import analyze

NEW_MIN_PCT = 0.2        # new avalanche area that makes a pair "positive" and raises a model alarm
orb, bf = cv2.ORB_create(5000), cv2.BFMatcher(cv2.NORM_HAMMING)
OUT = Path("/content/dl_results")
OUT.mkdir(exist_ok=True)


def homography(ref, mov):
    k1, d1 = orb.detectAndCompute(cv2.cvtColor(ref, cv2.COLOR_BGR2GRAY), None)
    k2, d2 = orb.detectAndCompute(cv2.cvtColor(mov, cv2.COLOR_BGR2GRAY), None)
    if d1 is None or d2 is None:
        return None, 0
    good = [p[0] for p in bf.knnMatch(d2, d1, k=2)
            if len(p) == 2 and p[0].distance < 0.75 * p[1].distance]
    if len(good) < 15:
        return None, 0
    src = np.float32([k2[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([k1[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, inl = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
    return H, (int(inl.sum()) if H is not None else 0)


def new_regions(later, earlier):
    """Pixels marked in the later mask that were not marked (nearby) in the earlier one."""
    old = cv2.dilate(earlier, np.ones((15, 15), np.uint8))
    new = ((later > 0) & (old == 0)).astype(np.uint8) * 255
    return cv2.morphologyEx(new, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))


def scores(pred, gt):
    hit = np.count_nonzero((pred > 0) & (gt > 0))
    near = cv2.dilate(gt, np.ones((31, 31), np.uint8))
    recall = hit / max(np.count_nonzero(gt), 1) * 100
    precision = np.count_nonzero((pred > 0) & (near > 0)) / max(np.count_nonzero(pred), 1) * 100
    return recall, precision


times = {it["key"]: it["time"] for it in idx["items"]}
rows = []
for view in idx["views"]:
    for a, b in zip(view, view[1:]):
        img_a, img_b = data[a][0], data[b][0]
        H, inliers = homography(img_a, img_b)
        if H is None or inliers < 60:
            continue
        h, w = img_a.shape[:2]
        gap_h = (datetime.strptime(times[b], "%Y%m%d%H%M")
                 - datetime.strptime(times[a], "%Y%m%d%H%M")).total_seconds() / 3600
        warp = lambda m: cv2.warpPerspective(m, H, (w, h), flags=cv2.INTER_NEAREST)
        gt = new_regions(warp((data[b][1] * 255).astype(np.uint8)), (data[a][1] * 255).astype(np.uint8))
        positive = np.count_nonzero(gt) / gt.size * 100 >= NEW_MIN_PCT

        with contextlib.redirect_stdout(io.StringIO()):
            r = analyze(img_a, img_b, roi=None, roi_file=None)
        classic = cv2.bitwise_or(r["change_mask"], r["motion_mask"])

        dl = new_regions(warp((prob[b] > 0.5).astype(np.uint8) * 255),
                         (prob[a] > 0.5).astype(np.uint8) * 255)
        dl_pct = np.count_nonzero(dl) / dl.size * 100
        rows.append({"pair": f"{a} -> {b}", "gap_h": round(gap_h, 1), "positive": bool(positive),
                     "classic_alarm": r["risk_level"] != "LOW", "classic_scores": scores(classic, gt),
                     "dl_alarm": bool(dl_pct >= NEW_MIN_PCT), "dl_new_pct": round(dl_pct, 2),
                     "dl_scores": scores(dl, gt)})

        # picture: later image, green = real new avalanches, red = model, blue = classical
        vis = cv2.warpPerspective(img_b, H, (w, h))
        for m, color in ((classic, (255, 120, 0)), (dl, (0, 0, 255)), (gt, (0, 255, 0))):
            cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(vis, cnts, -1, color, 3)
        cv2.imwrite(str(OUT / f"{a}__{b}.jpg"), cv2.resize(vis, None, fx=0.5, fy=0.5))


def summary(sel, label):
    pos = [r for r in sel if r["positive"]]
    neg = [r for r in sel if not r["positive"]]
    print(f"\n[{label}] {len(pos)} pairs with a new avalanche, {len(neg)} without")
    print(f"  {'method':<22}{'detected':>10}{'false alarms':>15}{'pixels found':>15}")
    for name in ("classic", "dl"):
        found = np.mean([r[name + "_scores"][0] for r in pos]) if pos else float("nan")
        print(f"  {'classical (repo)' if name == 'classic' else 'deep learning':<22}"
              f"{sum(r[name + '_alarm'] for r in pos):>6} / {len(pos):<3}"
              f"{sum(r[name + '_alarm'] for r in neg):>9} / {len(neg):<5}{found:>12.1f} %")


summary(rows, "all pairs")
summary([r for r in rows if r["gap_h"] <= 6], "pairs <= 6 hours apart")

In [ ]:
# 8) Download the results (summary + pictures) to put them into the repository
json.dump({"segmentation": seg, "pairs": rows, "folds": folds}, open(OUT / "results.json", "w"),
          indent=1, default=float)
!cd /content && zip -qr dl_results.zip dl_results
from google.colab import files
files.download("/content/dl_results.zip")